# Lab 6 - Governance

Implements Unity Catalog governance controls: permissions, row-level security, and column masking.

## Permissions

Grants the `account users` group access to the Gold schema.

**Note:** The GRANT statements below document the required permissions. These have already been applied to the workspace.

In [0]:
%sql
-- GRANT USE SCHEMA ON SCHEMA lab5.gold TO `account users`;

In [0]:
%sql
-- GRANT SELECT ON SCHEMA lab5.gold TO `account users`;

In [0]:
%sql
SHOW GRANTS ON SCHEMA lab5.gold;

## Row-Level Security

Restricts dim_customers to show only CA, NY, and TX states.

In [0]:
%sql
CREATE OR REPLACE FUNCTION lab5.gold.filter_customers_by_state(state STRING)
RETURNS BOOLEAN
RETURN state IN ('CA', 'NY', 'TX');

In [0]:
%sql
ALTER TABLE lab5.gold.dim_customers
SET ROW FILTER lab5.gold.filter_customers_by_state ON (state);

In [0]:
%sql
SELECT
    state,
    COUNT(*) AS customer_count
FROM lab5.gold.dim_customers
GROUP BY state
ORDER BY state;

## Column-Level Security

Masks tax_id for non-admin users.

In [0]:
%sql
CREATE OR REPLACE FUNCTION lab5.gold.mask_tax_id(value STRING)
RETURN CASE
    WHEN is_account_group_member('admins') THEN value
    ELSE 'REDACTED'
END;

In [0]:
%sql
ALTER TABLE lab5.gold.dim_customers
ALTER COLUMN tax_id
SET MASK lab5.gold.mask_tax_id;

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    tax_id
FROM lab5.gold.dim_customers
WHERE tax_id IS NOT NULL
LIMIT 10;